
# EF–ToM band-ERP, PSD and DE feature extraction

Start from the **interpolated epoch files**. This notebook extracts three families of participant-level EEG features:

1. **Band-limited evoked/ERP features** in theta, alpha, low-beta and beta.
2. **PSD** by condition × band × ROI.
3. **Differential entropy (DE)** by condition × band × ROI.

PSD/DE are kept **condition-specific** rather than automatically subtracting conditions. EF and ToM are processed independently, and ToM hearing/thinking are kept separate.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
import mne
from scipy.signal import welch

mne.set_log_level("WARNING")


## 1. Paths and feature definitions

In [2]:

BASE = Path(r"H:\ASD projoect")

# Change these for one matched participant
GROUP = "ASD"
EF_ID = "1170"
TOM_ID = "3200"

EF_DIR = BASE / "EF" / GROUP / f"{EF_ID}_CF_CV_processed_numpy"
TOM_DIR = BASE / "ToM" / GROUP / f"{TOM_ID}a_CV_processed_numpy"

EF_FILE = EF_DIR / "interpolated_epoch.npy"
TOM_HEARING_FILE = TOM_DIR / "interpolated_hearing_epoch.npy"
TOM_THINKING_FILE = TOM_DIR / "interpolated_thinking_epoch.npy"

BANDS = {
    "theta": (4, 7),
    "alpha": (8, 12),
    "low_beta": (12, 16),
    "beta": (16, 25),
}

ROIS = {
    "frontal": ["FP1","FPZ","FP2","AF3","AF4","F7","F5","F3","F1","FZ","F2","F4","F6","F8"],
    "frontocentral": ["FT7","FC5","FC3","FC1","FCZ","FC2","FC4","FC6","FT8"],
    "central": ["T7","C5","C3","C1","CZ","C2","C4","C6","T8"],
    "centroparietal": ["TP7","CP5","CP3","CP1","CPZ","CP2","CP4","CP6","TP8"],
    "parietal": ["P7","P5","P3","P1","PZ","P2","P4","P6","P8"],
    "posterior": ["PO7","PO5","PO3","POZ","PO4","PO6","PO8","O1","OZ","O2"],
}

print(EF_FILE)
print(TOM_HEARING_FILE)
print(TOM_THINKING_FILE)


H:\ASD projoect\EF\ASD\1170_CF_CV_processed_numpy\interpolated_epoch.npy
H:\ASD projoect\ToM\ASD\3200a_CV_processed_numpy\interpolated_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3200a_CV_processed_numpy\interpolated_thinking_epoch.npy


## 2. Load interpolated epochs

In [3]:

def load_interpolated(file_path):
    d = np.load(file_path, allow_pickle=True).item()
    meta = pd.DataFrame(d["metadata"])

    info = mne.create_info(
        ch_names=list(d["channels"]),
        sfreq=float(d["sfreq"]),
        ch_types="eeg",
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")

    ep = mne.EpochsArray(
        d["eeg"], info,
        tmin=float(d["times"][0]),
        metadata=meta,
        verbose=False,
    )
    ep.set_eeg_reference("average", projection=False, verbose=False)
    return ep


def get_conditions(epochs, task):
    m = epochs.metadata

    if task == "EF":
        g = m["go_nogo"].astype(str).str.lower()
        return {
            "go": (g == "go").to_numpy(),
            "nogo": (g == "nogo").to_numpy(),
        }

    if task == "ToM":
        tom = m["tom"].astype(str).str.lower()
        belief = m["belief"].astype(str).str.lower()
        return {
            "false_cog": ((belief == "false") & (tom == "cognitive")).to_numpy(),
            "true_cog":  ((belief == "true")  & (tom == "cognitive")).to_numpy(),
            "false_aff": ((belief == "false") & (tom == "affective")).to_numpy(),
            "true_aff":  ((belief == "true")  & (tom == "affective")).to_numpy(),
        }

    raise ValueError("task must be EF or ToM")



## 3. Band-limited evoked / ERP features

For each frequency band:

- band-pass filter the already-cleaned epochs,
- average trials within each condition,
- extract the evoked response from the established ERP ROIs/windows.

A signed mean of a narrow-band oscillatory waveform can cancel toward zero, so this notebook keeps only **RMS amplitude**.

EF:
- N2: FCz, 300–400 ms
- P3/LPP: Cz + CPz, 500–700 ms

ToM:
- LPC: frontal/frontocentral, 300–600 ms
- early LSW: posterior ROI, 800–1400 ms
- mid LSW: posterior ROI, 1400–2100 ms
- late LSW: posterior ROI, 2000–2500 ms


In [4]:

def get_erp_defs(task):
    if task == "EF":
        return {
            "N2": {
                "channels": ["FCZ"],
                "tmin": 0.300,
                "tmax": 0.400,
            },
            "P3_LPP": {
                "channels": ["CZ", "CPZ"],
                "tmin": 0.500,
                "tmax": 0.700,
            },
        }

    if task == "ToM":
        posterior = [
            "CPZ", "CP1", "CP2", "CP3", "CP4",
            "PZ", "P1", "P2", "P3", "P4",
            "POZ", "PO3", "PO4",
        ]

        return {
            "LPC": {
                "channels": ["FZ", "F1", "F2", "FCZ", "FC1", "FC2"],
                "tmin": 0.300,
                "tmax": 0.600,
            },
            "early_LSW": {
                "channels": posterior,
                "tmin": 0.800,
                "tmax": 1.400,
            },
            "mid_LSW": {
                "channels": posterior,
                "tmin": 1.400,
                "tmax": 2.100,
            },
            "late_LSW": {
                "channels": posterior,
                "tmin": 2.000,
                "tmax": 2.500,
            },
        }

    raise ValueError("task must be 'EF' or 'ToM'")


def band_erp_features(epochs, task):
    conditions = get_conditions(epochs, task)
    erp_defs = get_erp_defs(task)

    rows = []

    for band, (fmin, fmax) in BANDS.items():
        band_epochs = epochs.copy().filter(
            fmin,
            fmax,
            method="iir",
            iir_params=dict(order=4, ftype="butter"),
            verbose=False,
        )

        for condition, mask in conditions.items():
            idx = np.where(mask)[0]
            if len(idx) == 0:
                continue

            evoked = band_epochs[idx].average()

            for component, cfg in erp_defs.items():
                use_chs = [
                    ch for ch in cfg["channels"]
                    if ch in evoked.ch_names
                ]
                if len(use_chs) == 0:
                    continue

                x = (
                    evoked.copy()
                    .pick(use_chs)
                    .crop(cfg["tmin"], cfg["tmax"])
                    .data
                    * 1e6
                )

                rows.append({
                    "task": task,
                    "condition": condition,
                    "band": band,
                    "component": component,
                    "n_trials": len(idx),
                    "evoked_rms_uv": np.sqrt(np.mean(x ** 2)),
                })

    return pd.DataFrame(rows)



## 4. PSD + DE

PSD and DE are computed **per trial**, then averaged within condition.

- EF: Go and Nogo are retained separately.
- ToM: false-cognitive, true-cognitive, false-affective, true-affective are retained separately.
- No subtraction is forced at this stage.

DE is computed after band-pass filtering:

\[
DE=
rac12\ln(2\pi e\sigma^2)
\]

The DE signal is converted to µV before variance estimation so every participant uses the same numerical scale.


In [5]:

def roi_indices(ch_names):
    out = {}
    for roi, roi_chs in ROIS.items():
        idx = [ch_names.index(ch) for ch in roi_chs if ch in ch_names]
        if idx:
            out[roi] = idx
    return out


def analysis_interval(epochs, task):
    if task == "EF":
        return 0.0, min(1.0, float(epochs.times[-1]))
    if task == "ToM":
        return 0.0, min(2.5, float(epochs.times[-1]))
    raise ValueError


def psd_de_trial_features(epochs, task):
    conds = get_conditions(epochs, task)
    tmin, tmax = analysis_interval(epochs, task)
    ep = epochs.copy().crop(tmin=tmin, tmax=tmax)

    x_v = ep.get_data()              # V
    x_uv = x_v * 1e6                 # µV for DE
    sf = float(ep.info["sfreq"])
    rois = roi_indices(ep.ch_names)

    labels = np.array(["unassigned"] * len(ep), dtype=object)
    for cond, mask in conds.items():
        labels[mask] = cond

    nperseg = min(int(round(sf)), x_v.shape[-1])
    noverlap = nperseg // 2 if nperseg >= 4 else 0

    freqs, psd = welch(
        x_v,
        fs=sf,
        axis=-1,
        nperseg=nperseg,
        noverlap=noverlap,
        detrend="constant",
    )

    rows = []
    eps = 1e-30

    for band, (lo, hi) in BANDS.items():
        fm = (freqs >= lo) & (freqs <= hi)
        if not np.any(fm):
            continue

        # trial x channel
        bp = psd[..., fm].mean(axis=-1)

        bx = mne.filter.filter_data(
            x_uv,
            sfreq=sf,
            l_freq=lo,
            h_freq=hi,
            method="iir",
            iir_params=dict(order=4, ftype="butter"),
            verbose=False,
        )
        var = np.var(bx, axis=-1, ddof=1)
        de = 0.5 * np.log(2 * np.pi * np.e * np.maximum(var, eps))

        for trial in range(len(ep)):
            cond = labels[trial]
            if cond == "unassigned":
                continue

            for roi, idx in rois.items():
                roi_psd = float(np.mean(bp[trial, idx]))
                roi_de = float(np.mean(de[trial, idx]))

                rows.append({
                    "task": task,
                    "trial_idx": trial,
                    "condition": cond,
                    "band": band,
                    "roi": roi,
                    "psd_db": 10 * np.log10(max(roi_psd, eps)),
                    "de": roi_de,
                })

    return pd.DataFrame(rows)


def summarize_psd_de(trial_df):
    return (
        trial_df
        .groupby(["task","condition","band","roi"], as_index=False)
        .agg(
            psd_db=("psd_db","mean"),
            de=("de","mean"),
            n_trials=("trial_idx","nunique"),
        )
    )


## 5. Helpers for wide CCA-ready feature rows

In [6]:

def band_erp_to_wide(df, prefix):
    row = {}

    for _, r in df.iterrows():
        base = (
            f"{prefix}_bandERP_"
            f"{r['condition']}_"
            f"{r['band']}_"
            f"{r['component']}"
        )

        row[f"{base}_rms_uv"] = r["evoked_rms_uv"]

    return row


def psd_de_to_wide(df, prefix):
    row = {}

    for _, r in df.iterrows():
        base = (
            f"{prefix}_"
            f"{r['condition']}_"
            f"{r['band']}_"
            f"{r['roi']}"
        )

        row[f"{base}_PSD_dB"] = r["psd_db"]
        row[f"{base}_DE"] = r["de"]

    return row


## 6. Run one matched participant

In [7]:

# ----- EF -----
ef_epochs = load_interpolated(EF_FILE)
ef_band_erp = band_erp_features(ef_epochs, "EF")
ef_trial_pd = psd_de_trial_features(ef_epochs, "EF")
ef_pd = summarize_psd_de(ef_trial_pd)

# ----- ToM hearing -----
h_epochs = load_interpolated(TOM_HEARING_FILE)
h_band_erp = band_erp_features(h_epochs, "ToM")
h_trial_pd = psd_de_trial_features(h_epochs, "ToM")
h_pd = summarize_psd_de(h_trial_pd)

# ----- ToM thinking -----
t_epochs = load_interpolated(TOM_THINKING_FILE)
t_band_erp = band_erp_features(t_epochs, "ToM")
t_trial_pd = psd_de_trial_features(t_epochs, "ToM")
t_pd = summarize_psd_de(t_trial_pd)

print("EF band ERP:", ef_band_erp.shape)
print("EF PSD/DE:", ef_pd.shape)
print("Hearing band ERP:", h_band_erp.shape)
print("Hearing PSD/DE:", h_pd.shape)
print("Thinking band ERP:", t_band_erp.shape)
print("Thinking PSD/DE:", t_pd.shape)


EF band ERP: (16, 6)
EF PSD/DE: (48, 7)
Hearing band ERP: (64, 6)
Hearing PSD/DE: (96, 7)
Thinking band ERP: (64, 6)
Thinking PSD/DE: (96, 7)


## 7. Save one-participant outputs

In [8]:

# EF
ef_band_erp.to_csv(EF_DIR / "EF_bandERP_features.csv", index=False)
ef_trial_pd.to_csv(EF_DIR / "EF_PSD_DE_trial_features.csv", index=False)
ef_pd.to_csv(EF_DIR / "EF_PSD_DE_condition_features.csv", index=False)

# ToM hearing
h_band_erp.to_csv(TOM_DIR / "ToM_hearing_bandERP_features.csv", index=False)
h_trial_pd.to_csv(TOM_DIR / "ToM_hearing_PSD_DE_trial_features.csv", index=False)
h_pd.to_csv(TOM_DIR / "ToM_hearing_PSD_DE_condition_features.csv", index=False)

# ToM thinking
t_band_erp.to_csv(TOM_DIR / "ToM_thinking_bandERP_features.csv", index=False)
t_trial_pd.to_csv(TOM_DIR / "ToM_thinking_PSD_DE_trial_features.csv", index=False)
t_pd.to_csv(TOM_DIR / "ToM_thinking_PSD_DE_condition_features.csv", index=False)

row = {
    "group": GROUP,
    "EF_ID": EF_ID,
    "ToM_ID": TOM_ID,
    **band_erp_to_wide(ef_band_erp, "EF"),
    **psd_de_to_wide(ef_pd, "EF"),
    **band_erp_to_wide(h_band_erp, "H"),
    **psd_de_to_wide(h_pd, "H"),
    **band_erp_to_wide(t_band_erp, "T"),
    **psd_de_to_wide(t_pd, "T"),
}

wide = pd.DataFrame([row])
wide.to_csv(BASE / f"{GROUP}_{EF_ID}_{TOM_ID}_band_PSD_DE_features.csv", index=False)

print("Saved. Total columns:", wide.shape[1])


Saved. Total columns: 627


## 8. Optional batch processing for the currently matched participants

asd_sub_num_ef = [1261, 1371, 1441, 1271, 1241, 1301, 1541, 1461, 1501,
                  1391, 1701, 1411, 1561, 1611, 1631, 1591, 1731, 1601,
                  1661, 1780, 1331, 1401, 1291, 1220, 1251]

asd_sub_num_tom = [3571, 3231, 3421, 3091, 3211, 3181, 3141, 3241, 3361, 3131, 3601, 3281, 3031, 3501, 3541, 3491, 3611, 3581, 3461, 3450, 3411, 3221, 3151, 3250, 3341]

td_sub_num_ef = [2180, 2500, 2231, 2510, 2480, 2441, 2371, 2401, 2470,
                 2420, 2171, 2331, 2380, 2390, 2310, 2240, 2521, 2531,
                 2321, 2430, 2291, 2350, 2301]

td_sub_num_tom = [4260, 4430, 4281, 4390, 4380, 4441, 4421, 4331, 4480, 4320, 4231, 4451, 4400, 4410, 4160, 4110, 4471, 4511, 4351, 4370, 4301, 4531, 4461]

In [10]:

ASD_PAIRS = [
    # ("1170","3200"),("1161","3081"),("1121","3041"),("1101","3061"),
    # ("1141","3161"),("1151","3071"),("1191","3111"),("1031","3121"),
    # ("1021","3021"),("1011","3011"),("1071","3051"),
    ("1261","3571"),
    ("1371","3231"),("1441","3421"),("1271","3091"),("1241","3211"),
    ("1301","3181"),("1541","3141"),("1461","3241"),("1501","3361"),
    ("1391","3131"),("1701","3601"),("1411","3281"),("1561","3031"),
    ("1611","3501"),("1631","3541"),("1591","3491"),("1731","3611"),
    ("1601","3581"),("1661","3461"),("1780","3450"),("1331","3411"),
    ("1401","3221"),("1291","3151"),("1220","3250"),("1251","3341"),
]

TD_PAIRS = [
    # ("2360","4150"),("2121","4181"),("2201","4171"),("2250","4210"),
    # ("2281","4091"),("2340","4080"),("2151","4071"),("2211","4141"),
    ("2180","4260"),("2500","4430"),("2231","4281"),("2510","4390"),
    ("2480","4380"),("2441","4441"),("2371","4421"),("2401","4331"),
    ("2470","4480"),("2420","4320"),("2171","4231"),("2331","4451"),
    ("2380","4400"),("2390","4410"),("2310","4160"),("2240","4110"),
    ("2521","4471"),("2531","4511"),("2321","4351"),("2430","4370"),
    ("2291","4301"),("2350","4531"),("2301","4461"),
]


def extract_one_file(file_path, task, prefix):
    ep = load_interpolated(file_path)
    be = band_erp_features(ep, task)
    tr = psd_de_trial_features(ep, task)
    pd_sum = summarize_psd_de(tr)
    wide = {
        **band_erp_to_wide(be, prefix),
        **psd_de_to_wide(pd_sum, prefix),
    }
    return be, tr, pd_sum, wide


def process_group(group, pair_list):
    rows = []

    for ef_id, tom_id in pair_list:
        print(f"{group}: EF {ef_id} / ToM {tom_id}")

        ef_dir = BASE / "EF" / group / f"{ef_id}_CF_CV_processed_numpy"
        tom_dir = BASE / "ToM" / group / f"{tom_id}a_CV_processed_numpy"

        ef_file = ef_dir / "interpolated_epoch.npy"
        h_file = tom_dir / "interpolated_hearing_epoch.npy"
        t_file = tom_dir / "interpolated_thinking_epoch.npy"

        if not (ef_file.exists() and h_file.exists() and t_file.exists()):
            print("  missing interpolated file -> skipped")
            continue

        ef_be, ef_tr, ef_pd, ef_w = extract_one_file(ef_file, "EF", "EF")
        h_be, h_tr, h_pd, h_w = extract_one_file(h_file, "ToM", "H")
        t_be, t_tr, t_pd, t_w = extract_one_file(t_file, "ToM", "T")

        ef_be.to_csv(ef_dir / "EF_bandERP_features.csv", index=False)
        ef_tr.to_csv(ef_dir / "EF_PSD_DE_trial_features.csv", index=False)
        ef_pd.to_csv(ef_dir / "EF_PSD_DE_condition_features.csv", index=False)

        h_be.to_csv(tom_dir / "ToM_hearing_bandERP_features.csv", index=False)
        h_tr.to_csv(tom_dir / "ToM_hearing_PSD_DE_trial_features.csv", index=False)
        h_pd.to_csv(tom_dir / "ToM_hearing_PSD_DE_condition_features.csv", index=False)

        t_be.to_csv(tom_dir / "ToM_thinking_bandERP_features.csv", index=False)
        t_tr.to_csv(tom_dir / "ToM_thinking_PSD_DE_trial_features.csv", index=False)
        t_pd.to_csv(tom_dir / "ToM_thinking_PSD_DE_condition_features.csv", index=False)

        rows.append({
            "group": group,
            "EF_ID": ef_id,
            "ToM_ID": tom_id,
            **ef_w, **h_w, **t_w,
        })

    return pd.DataFrame(rows)


In [11]:

# Run only after testing one participant above.
#
ASD_features = process_group("ASD", ASD_PAIRS)
TD_features = process_group("TD", TD_PAIRS)

ASD_features.to_csv(BASE / "ASD_band_PSD_DE_features.csv", index=False)
TD_features.to_csv(BASE / "TD_band_PSD_DE_features.csv", index=False)


ASD: EF 1261 / ToM 3571
ASD: EF 1371 / ToM 3231
ASD: EF 1441 / ToM 3421
ASD: EF 1271 / ToM 3091
ASD: EF 1241 / ToM 3211
ASD: EF 1301 / ToM 3181
ASD: EF 1541 / ToM 3141
ASD: EF 1461 / ToM 3241
  missing interpolated file -> skipped
ASD: EF 1501 / ToM 3361
ASD: EF 1391 / ToM 3131
ASD: EF 1701 / ToM 3601
  missing interpolated file -> skipped
ASD: EF 1411 / ToM 3281
ASD: EF 1561 / ToM 3031
ASD: EF 1611 / ToM 3501
ASD: EF 1631 / ToM 3541
ASD: EF 1591 / ToM 3491
ASD: EF 1731 / ToM 3611
  missing interpolated file -> skipped
ASD: EF 1601 / ToM 3581
ASD: EF 1661 / ToM 3461
ASD: EF 1780 / ToM 3450
ASD: EF 1331 / ToM 3411
ASD: EF 1401 / ToM 3221
ASD: EF 1291 / ToM 3151
ASD: EF 1220 / ToM 3250
ASD: EF 1251 / ToM 3341
TD: EF 2180 / ToM 4260
TD: EF 2500 / ToM 4430
  missing interpolated file -> skipped
TD: EF 2231 / ToM 4281
TD: EF 2510 / ToM 4390
TD: EF 2480 / ToM 4380
TD: EF 2441 / ToM 4441
TD: EF 2371 / ToM 4421
TD: EF 2401 / ToM 4331
TD: EF 2470 / ToM 4480
TD: EF 2420 / ToM 4320
TD: EF 2171 / 


## Before CCA

The wide tables are deliberately high-dimensional. With the current matched sample, do **not** run ordinary CCA yet. After preprocessing more participants, the next step should be feature QC, standardization, and either dimensionality reduction or regularized CCA.



# 11. EF–ToM correlation testing

To avoid an all-to-all feature search, the primary analysis is constrained.

**PSD / DE**
- EF: Nogo trials
- ToM: false-cognitive and false-affective trials
- correlate only the same frequency band and the same ROI across tasks

**Band-limited ERP**
- keep RMS only
- EF N2 ↔ ToM LPC
- EF P3/LPP ↔ ToM early/mid/late LSW

ASD and TD are analyzed separately. Hearing and thinking are analyzed separately.


In [13]:

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests


In [14]:

def safe_spearman(x, y):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")

    valid = x.notna() & y.notna()
    n = int(valid.sum())

    if n < 4:
        return n, np.nan, np.nan

    rho, p = spearmanr(
        x[valid],
        y[valid],
    )

    return n, rho, p


def add_fdr(result_df):
    result_df = result_df.copy()
    result_df["p_FDR"] = np.nan

    valid = result_df["p"].notna()

    if valid.sum() > 0:
        result_df.loc[valid, "p_FDR"] = multipletests(
            result_df.loc[valid, "p"],
            method="fdr_bh",
        )[1]

    return result_df.sort_values(
        ["p_FDR", "p"],
        na_position="last",
    ).reset_index(drop=True)


## 11.1 PSD / DE: same band + same ROI

In [15]:

def spectral_cross_task_corr(
    df,
    tom_prefix,
    tom_condition,
    metric,
):
    # metric: "PSD_dB" or "DE"
    # tom_prefix: "H" (hearing) or "T" (thinking)
    # tom_condition: "false_cog" or "false_aff"

    rows = []

    for band in BANDS:
        for roi in ROIS:

            ef_col = f"EF_nogo_{band}_{roi}_{metric}"
            tom_col = (
                f"{tom_prefix}_{tom_condition}_"
                f"{band}_{roi}_{metric}"
            )

            if ef_col not in df.columns or tom_col not in df.columns:
                continue

            n, rho, p = safe_spearman(
                df[ef_col],
                df[tom_col],
            )

            rows.append({
                "band": band,
                "roi": roi,
                "EF_feature": ef_col,
                "ToM_feature": tom_col,
                "n": n,
                "rho": rho,
                "p": p,
            })

    return add_fdr(pd.DataFrame(rows))


In [16]:

def run_spectral_correlations(df, group_name):
    outputs = {}

    for segment_name, prefix in [
        ("hearing", "H"),
        ("thinking", "T"),
    ]:
        for tom_condition in [
            "false_cog",
            "false_aff",
        ]:
            for metric in [
                "PSD_dB",
                "DE",
            ]:

                key = (
                    f"{group_name}_"
                    f"{segment_name}_"
                    f"{tom_condition}_"
                    f"{metric}"
                )

                outputs[key] = spectral_cross_task_corr(
                    df=df,
                    tom_prefix=prefix,
                    tom_condition=tom_condition,
                    metric=metric,
                )

    return outputs


## 11.2 Band-limited ERP: theory-guided component pairs

In [17]:

ERP_PAIRINGS = [
    ("N2", "LPC"),
    ("P3_LPP", "early_LSW"),
    ("P3_LPP", "mid_LSW"),
    ("P3_LPP", "late_LSW"),
]


def banderp_cross_task_corr(
    df,
    tom_prefix,
    tom_condition,
):
    rows = []

    for band in BANDS:
        for ef_component, tom_component in ERP_PAIRINGS:

            ef_col = (
                f"EF_bandERP_nogo_"
                f"{band}_{ef_component}_rms_uv"
            )

            tom_col = (
                f"{tom_prefix}_bandERP_"
                f"{tom_condition}_{band}_"
                f"{tom_component}_rms_uv"
            )

            if ef_col not in df.columns or tom_col not in df.columns:
                continue

            n, rho, p = safe_spearman(
                df[ef_col],
                df[tom_col],
            )

            rows.append({
                "band": band,
                "EF_component": ef_component,
                "ToM_component": tom_component,
                "EF_feature": ef_col,
                "ToM_feature": tom_col,
                "n": n,
                "rho": rho,
                "p": p,
            })

    return add_fdr(pd.DataFrame(rows))


In [18]:

def run_banderp_correlations(df, group_name):
    outputs = {}

    for segment_name, prefix in [
        ("hearing", "H"),
        ("thinking", "T"),
    ]:
        for tom_condition in [
            "false_cog",
            "false_aff",
        ]:

            key = (
                f"{group_name}_"
                f"{segment_name}_"
                f"{tom_condition}_"
                f"bandERP"
            )

            outputs[key] = banderp_cross_task_corr(
                df=df,
                tom_prefix=prefix,
                tom_condition=tom_condition,
            )

    return outputs


## 11.3 Load group feature tables

In [19]:

ASD_FEATURE_FILE = BASE / "ASD_band_PSD_DE_features.csv"
TD_FEATURE_FILE = BASE / "TD_band_PSD_DE_features.csv"

ASD_features = pd.read_csv(
    ASD_FEATURE_FILE
)

TD_features = pd.read_csv(
    TD_FEATURE_FILE
)

print("ASD:", ASD_features.shape)
print("TD:", TD_features.shape)


ASD: (22, 627)
TD: (21, 627)


## 11.4 Run correlations

In [20]:

ASD_spectral_corr = run_spectral_correlations(
    ASD_features,
    "ASD",
)

TD_spectral_corr = run_spectral_correlations(
    TD_features,
    "TD",
)

ASD_banderp_corr = run_banderp_correlations(
    ASD_features,
    "ASD",
)

TD_banderp_corr = run_banderp_correlations(
    TD_features,
    "TD",
)


In [21]:

# Examples
display(
    ASD_spectral_corr[
        "ASD_hearing_false_cog_PSD_dB"
    ].head(10)
)

display(
    ASD_banderp_corr[
        "ASD_hearing_false_cog_bandERP"
    ].head(10)
)


,band,roi,EF_feature,ToM_feature,n,rho,p,p_FDR
0,low_beta,centroparietal,EF_nogo_low_beta_centroparietal_PSD_dB,H_false_cog_low_beta_centroparietal_PSD_dB,22,0.894975,1.898366e-08,4.556078e-07
1,low_beta,parietal,EF_nogo_low_beta_parietal_PSD_dB,H_false_cog_low_beta_parietal_PSD_dB,22,0.871259,1.313049e-07,1.575658e-06
2,theta,central,EF_nogo_theta_central_PSD_dB,H_false_cog_theta_central_PSD_dB,22,0.853190,4.514262e-07,3.611409e-06
3,low_beta,central,EF_nogo_low_beta_central_PSD_dB,H_false_cog_low_beta_central_PSD_dB,22,0.836251,1.249208e-06,7.495245e-06
4,low_beta,frontocentral,EF_nogo_low_beta_frontocentral_PSD_dB,H_false_cog_low_beta_frontocentral_PSD_dB,22,0.827216,2.053981e-06,9.859110e-06
5,theta,parietal,EF_nogo_theta_parietal_PSD_dB,H_false_cog_theta_parietal_PSD_dB,22,0.808018,5.410463e-06,2.164185e-05
6,theta,posterior,EF_nogo_theta_posterior_PSD_dB,H_false_cog_theta_posterior_PSD_dB,22,0.797854,8.661460e-06,2.969644e-05
7,theta,centroparietal,EF_nogo_theta_centroparietal_PSD_dB,H_false_cog_theta_centroparietal_PSD_dB,22,0.794466,1.007379e-05,3.022138e-05
8,theta,frontocentral,EF_nogo_theta_frontocentral_PSD_dB,H_false_cog_theta_frontocentral_PSD_dB,22,0.722191,1.475631e-04,3.577878e-04
9,beta,frontocentral,EF_nogo_beta_frontocentral_PSD_dB,H_false_cog_beta_frontocentral_PSD_dB,22,0.721062,1.528714e-04,3.577878e-04


,band,EF_component,ToM_component,EF_feature,ToM_feature,n,rho,p,p_FDR
0,alpha,N2,LPC,EF_bandERP_nogo_alpha_N2_rms_uv,H_bandERP_false_cog_alpha_LPC_rms_uv,22,0.495200,0.019114,0.126000
1,alpha,P3_LPP,mid_LSW,EF_bandERP_nogo_alpha_P3_LPP_rms_uv,H_bandERP_false_cog_alpha_mid_LSW_rms_uv,22,0.479390,0.023975,0.126000
2,alpha,P3_LPP,early_LSW,EF_bandERP_nogo_alpha_P3_LPP_rms_uv,H_bandERP_false_cog_alpha_early_LSW_rms_uv,22,0.472614,0.026340,0.126000
3,theta,P3_LPP,mid_LSW,EF_bandERP_nogo_theta_P3_LPP_rms_uv,H_bandERP_false_cog_theta_mid_LSW_rms_uv,22,0.453416,0.034061,0.126000
4,theta,P3_LPP,early_LSW,EF_bandERP_nogo_theta_P3_LPP_rms_uv,H_bandERP_false_cog_theta_early_LSW_rms_uv,22,0.442123,0.039375,0.126000
5,beta,P3_LPP,early_LSW,EF_bandERP_nogo_beta_P3_LPP_rms_uv,H_bandERP_false_cog_beta_early_LSW_rms_uv,22,0.409373,0.058501,0.156003
6,beta,P3_LPP,late_LSW,EF_bandERP_nogo_beta_P3_LPP_rms_uv,H_bandERP_false_cog_beta_late_LSW_rms_uv,22,0.367589,0.092372,0.183841
7,theta,N2,LPC,EF_bandERP_nogo_theta_N2_rms_uv,H_bandERP_false_cog_theta_LPC_rms_uv,22,0.354037,0.105985,0.183841
8,beta,P3_LPP,mid_LSW,EF_bandERP_nogo_beta_P3_LPP_rms_uv,H_bandERP_false_cog_beta_mid_LSW_rms_uv,22,0.351779,0.108389,0.183841
9,alpha,P3_LPP,late_LSW,EF_bandERP_nogo_alpha_P3_LPP_rms_uv,H_bandERP_false_cog_alpha_late_LSW_rms_uv,22,0.341615,0.119702,0.183841


## 11.5 Save correlation results

In [22]:

CORR_DIR = BASE / "EF_ToM_correlation_results_20"
CORR_DIR.mkdir(exist_ok=True)

all_outputs = {
    **ASD_spectral_corr,
    **TD_spectral_corr,
    **ASD_banderp_corr,
    **TD_banderp_corr,
}

for name, result in all_outputs.items():
    result.to_csv(
        CORR_DIR / f"{name}.csv",
        index=False,
    )

print("Saved to:", CORR_DIR)


Saved to: H:\ASD projoect\EF_ToM_correlation_results_20


## 11.6 Compact exploratory summary

In [23]:

summary_rows = []

for name, result in all_outputs.items():

    if len(result) == 0:
        continue

    temp = result.copy()
    temp["analysis"] = name
    summary_rows.append(temp)

correlation_summary = pd.concat(
    summary_rows,
    ignore_index=True,
    sort=False,
)

correlation_summary = correlation_summary.sort_values(
    ["p_FDR", "p"],
    na_position="last",
)

display(
    correlation_summary.head(30)
)

correlation_summary.to_csv(
    CORR_DIR / "all_correlation_results_summary_20.csv",
    index=False,
)


,band,roi,EF_feature,ToM_feature,n,rho,p,p_FDR,analysis,EF_component,ToM_component
216,alpha,frontocentral,EF_nogo_alpha_frontocentral_DE,H_false_cog_alpha_frontocentral_DE,21,0.975325,6.334485e-14,1.520276e-12,TD_hearing_false_cog_DE,NaN,NaN
360,low_beta,parietal,EF_nogo_low_beta_parietal_DE,T_false_aff_low_beta_parietal_DE,21,0.948052,6.711682e-11,1.610804e-09,TD_thinking_false_aff_DE,NaN,NaN
217,alpha,central,EF_nogo_alpha_central_DE,H_false_cog_alpha_central_DE,21,0.940260,2.455475e-10,1.964380e-09,TD_hearing_false_cog_DE,NaN,NaN
218,beta,parietal,EF_nogo_beta_parietal_DE,H_false_cog_beta_parietal_DE,21,0.940260,2.455475e-10,1.964380e-09,TD_hearing_false_cog_DE,NaN,NaN
192,alpha,frontocentral,EF_nogo_alpha_frontocentral_PSD_dB,H_false_cog_alpha_frontocentral_PSD_dB,21,0.946753,8.442873e-11,2.026290e-09,TD_hearing_false_cog_PSD_dB,NaN,NaN
193,alpha,centroparietal,EF_nogo_alpha_centroparietal_PSD_dB,H_false_cog_alpha_centroparietal_PSD_dB,21,0.941558,2.002991e-10,2.403589e-09,TD_hearing_false_cog_PSD_dB,NaN,NaN
219,alpha,parietal,EF_nogo_alpha_parietal_DE,H_false_cog_alpha_parietal_DE,21,0.936364,4.406761e-10,2.644056e-09,TD_hearing_false_cog_DE,NaN,NaN
220,alpha,centroparietal,EF_nogo_alpha_centroparietal_DE,H_false_cog_alpha_centroparietal_DE,21,0.933766,6.378436e-10,3.061650e-09,TD_hearing_false_cog_DE,NaN,NaN
221,low_beta,parietal,EF_nogo_low_beta_parietal_DE,H_false_cog_low_beta_parietal_DE,21,0.931169,9.098173e-10,3.639269e-09,TD_hearing_false_cog_DE,NaN,NaN
264,low_beta,parietal,EF_nogo_low_beta_parietal_DE,H_false_aff_low_beta_parietal_DE,21,0.941558,2.002991e-10,4.807178e-09,TD_hearing_false_aff_DE,NaN,NaN


# New: Re-run for other conditions

In [13]:

def safe_spearman(x, y):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")

    valid = x.notna() & y.notna()
    n = int(valid.sum())

    if n < 4:
        return n, np.nan, np.nan

    rho, p = spearmanr(
        x[valid],
        y[valid],
    )

    return n, rho, p


def add_fdr(result_df):
    result_df = result_df.copy()
    result_df["p_FDR"] = np.nan

    valid = result_df["p"].notna()

    if valid.sum() > 0:
        result_df.loc[valid, "p_FDR"] = multipletests(
            result_df.loc[valid, "p"],
            method="fdr_bh",
        )[1]

    return result_df.sort_values(
        ["p_FDR", "p"],
        na_position="last",
    ).reset_index(drop=True)


In [ ]:

def spectral_cross_task_corr(
    df,
    tom_prefix,
    tom_condition,
    metric,
):
    # metric: "PSD_dB" or "DE"
    # tom_prefix: "H" (hearing) or "T" (thinking)
    # tom_condition: "false_cog" or "false_aff"

    rows = []

    for band in BANDS:
        for roi in ROIS:

            ef_col = f"EF_nogo_{band}_{roi}_{metric}"
            tom_col = (
                f"{tom_prefix}_{tom_condition}_"
                f"{band}_{roi}_{metric}"
            )

            if ef_col not in df.columns or tom_col not in df.columns:
                continue

            n, rho, p = safe_spearman(
                df[ef_col],
                df[tom_col],
            )

            rows.append({
                "band": band,
                "roi": roi,
                "EF_feature": ef_col,
                "ToM_feature": tom_col,
                "n": n,
                "rho": rho,
                "p": p,
            })

    return add_fdr(pd.DataFrame(rows))


In [1]:
# 1. Re-run PSD / DE correlations using ALL four ToM conditions

TOM_CONDITIONS = [
    "false_cog",
    "true_cog",
    "false_aff",
    "true_aff",
]


def run_spectral_correlations_all(df, group_name):

    outputs = {}

    for segment_name, prefix in [
        ("hearing", "H"),
        ("thinking", "T"),
    ]:

        for condition in TOM_CONDITIONS:

            for metric in [
                "PSD_dB",
                "DE",
            ]:

                key = (
                    f"{group_name}_"
                    f"{segment_name}_"
                    f"{condition}_"
                    f"{metric}"
                )

                outputs[key] = spectral_cross_task_corr(
                    df=df,
                    tom_prefix=prefix,
                    tom_condition=condition,
                    metric=metric,
                )

    return outputs

In [15]:

ERP_PAIRINGS = [
    ("N2", "LPC"),
    ("P3_LPP", "early_LSW"),
    ("P3_LPP", "mid_LSW"),
    ("P3_LPP", "late_LSW"),
]


def banderp_cross_task_corr(
    df,
    tom_prefix,
    tom_condition,
):
    rows = []

    for band in BANDS:
        for ef_component, tom_component in ERP_PAIRINGS:

            ef_col = (
                f"EF_bandERP_nogo_"
                f"{band}_{ef_component}_rms_uv"
            )

            tom_col = (
                f"{tom_prefix}_bandERP_"
                f"{tom_condition}_{band}_"
                f"{tom_component}_rms_uv"
            )

            if ef_col not in df.columns or tom_col not in df.columns:
                continue

            n, rho, p = safe_spearman(
                df[ef_col],
                df[tom_col],
            )

            rows.append({
                "band": band,
                "EF_component": ef_component,
                "ToM_component": tom_component,
                "EF_feature": ef_col,
                "ToM_feature": tom_col,
                "n": n,
                "rho": rho,
                "p": p,
            })

    return add_fdr(pd.DataFrame(rows))


In [2]:
# 2. Re-run band-ERP correlations using ALL four ToM conditions

def run_banderp_correlations_all(df, group_name):

    outputs = {}

    for segment_name, prefix in [
        ("hearing", "H"),
        ("thinking", "T"),
    ]:

        for condition in TOM_CONDITIONS:

            key = (
                f"{group_name}_"
                f"{segment_name}_"
                f"{condition}_"
                f"bandERP"
            )

            outputs[key] = banderp_cross_task_corr(
                df=df,
                tom_prefix=prefix,
                tom_condition=condition,
            )

    return outputs

In [24]:
from pathlib import Path
import pandas as pd

BASE = Path(r"H:\ASD projoect")

ASD_FEATURE_FILE = (
    BASE / "ASD_band_PSD_DE_features.csv"
)

TD_FEATURE_FILE = (
    BASE / "TD_band_PSD_DE_features.csv"
)

ASD_features = pd.read_csv(
    ASD_FEATURE_FILE
)

TD_features = pd.read_csv(
    TD_FEATURE_FILE
)

print("ASD:", ASD_features.shape)
print("TD:", TD_features.shape)

ASD: (22, 627)
TD: (21, 627)


In [16]:
# 3. Run ASD and TD separately

ASD_spectral_all = run_spectral_correlations_all(
    ASD_features,
    "ASD"
)

TD_spectral_all = run_spectral_correlations_all(
    TD_features,
    "TD"
)

ASD_banderp_all = run_banderp_correlations_all(
    ASD_features,
    "ASD"
)

TD_banderp_all = run_banderp_correlations_all(
    TD_features,
    "TD"
)

In [18]:
# 5. Put false and true correlations side-by-side
#    for the SAME band + ROI

def compare_false_true_spectral(
    outputs,
    group,
    segment,
    domain,       # "cog" or "aff"
    metric,       # "PSD_dB" or "DE"
):

    false_df = outputs[
        f"{group}_{segment}_false_{domain}_{metric}"
    ].copy()

    true_df = outputs[
        f"{group}_{segment}_true_{domain}_{metric}"
    ].copy()

    false_df = false_df[
        ["band", "roi", "rho", "p", "p_FDR"]
    ].rename(columns={
        "rho": "rho_false",
        "p": "p_false",
        "p_FDR": "p_FDR_false"
    })

    true_df = true_df[
        ["band", "roi", "rho", "p", "p_FDR"]
    ].rename(columns={
        "rho": "rho_true",
        "p": "p_true",
        "p_FDR": "p_FDR_true"
    })

    result = false_df.merge(
        true_df,
        on=["band", "roi"],
        how="inner"
    )

    result["rho_difference"] = (
        result["rho_false"]
        - result["rho_true"]
    )

    return result.sort_values(
        "rho_difference",
        key=abs,
        ascending=False
    ).reset_index(drop=True)

# ERP tests

In [26]:
BANDS = [
    "theta",
    "alpha",
    "low_beta",
    "beta"
]

EF_CONDITIONS = [
    "go",
    "nogo"
]

TOM_CONDITIONS = [
    "false_cog",
    "true_cog",
    "false_aff",
    "true_aff"
]

ERP_PAIRINGS = [
    ("N2", "LPC"),
    ("P3_LPP", "early_LSW"),
    ("P3_LPP", "mid_LSW"),
    ("P3_LPP", "late_LSW"),
]

In [27]:
def calculate_all_erp_relations(df, group):

    results = []

    for segment, prefix in [
        ("hearing", "H"),
        ("thinking", "T")
    ]:

        for ef_condition in EF_CONDITIONS:

            for tom_condition in TOM_CONDITIONS:

                for band in BANDS:

                    for ef_comp, tom_comp in ERP_PAIRINGS:

                        ef_col = (
                            f"EF_bandERP_"
                            f"{ef_condition}_"
                            f"{band}_"
                            f"{ef_comp}_rms_uv"
                        )

                        tom_col = (
                            f"{prefix}_bandERP_"
                            f"{tom_condition}_"
                            f"{band}_"
                            f"{tom_comp}_rms_uv"
                        )

                        if (
                            ef_col not in df.columns
                            or tom_col not in df.columns
                        ):
                            continue

                        x = pd.to_numeric(
                            df[ef_col],
                            errors="coerce"
                        )

                        y = pd.to_numeric(
                            df[tom_col],
                            errors="coerce"
                        )

                        valid = (
                            x.notna()
                            & y.notna()
                        )

                        if valid.sum() < 4:
                            continue

                        rho, p = spearmanr(
                            x[valid],
                            y[valid]
                        )

                        results.append({
                            "group": group,
                            "segment": segment,
                            "EF_condition": ef_condition,
                            "ToM_condition": tom_condition,
                            "band": band,
                            "EF_component": ef_comp,
                            "ToM_component": tom_comp,
                            "n": valid.sum(),
                            "rho": rho,
                            "p": p
                        })

    results = pd.DataFrame(results)

    return results

In [28]:
ASD_ERP_results = calculate_all_erp_relations(
    ASD_features,
    "ASD"
)

TD_ERP_results = calculate_all_erp_relations(
    TD_features,
    "TD"
)

print("ASD tests:", len(ASD_ERP_results))
print("TD tests:", len(TD_ERP_results))

ASD tests: 256
TD tests: 256


In [29]:
ASD_ERP_results["p_FDR"] = multipletests(
    ASD_ERP_results["p"],
    method="fdr_bh"
)[1]

TD_ERP_results["p_FDR"] = multipletests(
    TD_ERP_results["p"],
    method="fdr_bh"
)[1]

In [30]:
save_dir = (
    BASE
    / "EF_ToM_ERP_relation"
)

save_dir.mkdir(
    exist_ok=True
)

ASD_ERP_results.to_csv(
    save_dir
    / "ASD_ERP_all_conditions.csv",
    index=False
)

TD_ERP_results.to_csv(
    save_dir
    / "TD_ERP_all_conditions.csv",
    index=False
)

print("Saved.")

Saved.


# PSD test
using PSD_ROI - PSD_all to make sure we clear the infuluence from each individual. We focus on rask-ralated feature rather than individual-related one.

In [25]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import linregress, spearmanr
from statsmodels.stats.multitest import multipletests

BASE = Path(r"H:\ASD projoect")

ASD = pd.read_csv(
    BASE / "ASD_band_PSD_DE_features.csv"
)

TD = pd.read_csv(
    BASE / "TD_band_PSD_DE_features.csv"
)

print("ASD:", ASD.shape)
print("TD:", TD.shape)

ASD: (22, 627)
TD: (21, 627)


In [26]:
BANDS = [
    "theta",
    "alpha",
    "low_beta",
    "beta"
]

ROIS = [
    "frontal",
    "frontocentral",
    "central",
    "centroparietal",
    "parietal",
    "posterior"
]

In [27]:
def center_psd(df, prefix, condition):

    df = df.copy()

    for band in BANDS:

        cols = [
            f"{prefix}_{condition}_{band}_{roi}_PSD_dB"
            for roi in ROIS
        ]

        available = [
            c for c in cols
            if c in df.columns
        ]

        global_mean = df[available].mean(axis=1)

        for roi in ROIS:

            col = (
                f"{prefix}_{condition}_"
                f"{band}_{roi}_PSD_dB"
            )

            if col not in df.columns:
                continue

            df[
                f"{prefix}_{condition}_"
                f"{band}_{roi}_PSD_centered"
            ] = (
                df[col]
                - global_mean
            )

    return df

In [28]:
def prepare_centered_psd(df):

    # EF
    for condition in ["go", "nogo"]:
        df = center_psd(
            df,
            "EF",
            condition
        )

    # ToM hearing
    for condition in [
        "false_cog",
        "true_cog",
        "false_aff",
        "true_aff"
    ]:
        df = center_psd(
            df,
            "H",
            condition
        )

    # ToM thinking
    for condition in [
        "false_cog",
        "true_cog",
        "false_aff",
        "true_aff"
    ]:
        df = center_psd(
            df,
            "T",
            condition
        )

    return df


ASD = prepare_centered_psd(ASD)
TD = prepare_centered_psd(TD)

In [29]:
def residualize(target, control):

    target = pd.to_numeric(
        target,
        errors="coerce"
    )

    control = pd.to_numeric(
        control,
        errors="coerce"
    )

    valid = (
        target.notna()
        & control.notna()
    )

    residual = pd.Series(
        np.nan,
        index=target.index
    )

    if valid.sum() < 4:
        return residual

    slope, intercept, _, _, _ = linregress(
        control[valid],
        target[valid]
    )

    residual.loc[valid] = (
        target[valid]
        - (
            intercept
            + slope * control[valid]
        )
    )

    return residual

In [30]:
def add_EF_residuals(df):

    for band in BANDS:

        for roi in ROIS:

            nogo = (
                f"EF_nogo_{band}_"
                f"{roi}_PSD_centered"
            )

            go = (
                f"EF_go_{band}_"
                f"{roi}_PSD_centered"
            )

            if nogo not in df.columns or go not in df.columns:
                continue

            df[
                f"EF_specific_{band}_{roi}"
            ] = residualize(
                df[nogo],
                df[go]
            )

    return df

In [31]:
def add_ToM_residuals(df):

    for prefix in ["H", "T"]:

        for domain in ["cog", "aff"]:

            for band in BANDS:

                for roi in ROIS:

                    false = (
                        f"{prefix}_false_{domain}_"
                        f"{band}_{roi}_PSD_centered"
                    )

                    true = (
                        f"{prefix}_true_{domain}_"
                        f"{band}_{roi}_PSD_centered"
                    )

                    if (
                        false not in df.columns
                        or true not in df.columns
                    ):
                        continue

                    df[
                        f"{prefix}_specific_{domain}_"
                        f"{band}_{roi}"
                    ] = residualize(
                        df[false],
                        df[true]
                    )

    return df

In [32]:
ASD = add_EF_residuals(ASD)
ASD = add_ToM_residuals(ASD)

TD = add_EF_residuals(TD)
TD = add_ToM_residuals(TD)

In [33]:
def correlate_specific_psd(
    df,
    segment,
    domain
):

    # segment = "H" or "T"
    # domain = "cog" or "aff"

    results = []

    for band in BANDS:

        for roi in ROIS:

            ef_col = (
                f"EF_specific_"
                f"{band}_{roi}"
            )

            tom_col = (
                f"{segment}_specific_"
                f"{domain}_{band}_{roi}"
            )

            if (
                ef_col not in df.columns
                or tom_col not in df.columns
            ):
                continue

            x = df[ef_col]
            y = df[tom_col]

            valid = (
                x.notna()
                & y.notna()
            )

            if valid.sum() < 4:
                continue

            rho, p = spearmanr(
                x[valid],
                y[valid]
            )

            results.append({
                "band": band,
                "roi": roi,
                "n": valid.sum(),
                "rho": rho,
                "p": p
            })

    results = pd.DataFrame(results)

    if len(results) > 0:

        results["p_FDR"] = multipletests(
            results["p"],
            method="fdr_bh"
        )[1]

    return results.sort_values(
        "p"
    ).reset_index(drop=True)

In [34]:
ASD_H_cog = correlate_specific_psd(
    ASD,
    segment="H",
    domain="cog"
)

ASD_H_aff = correlate_specific_psd(
    ASD,
    segment="H",
    domain="aff"
)

ASD_T_cog = correlate_specific_psd(
    ASD,
    segment="T",
    domain="cog"
)

ASD_T_aff = correlate_specific_psd(
    ASD,
    segment="T",
    domain="aff"
)

In [35]:
TD_H_cog = correlate_specific_psd(
    TD,
    segment="H",
    domain="cog"
)

TD_H_aff = correlate_specific_psd(
    TD,
    segment="H",
    domain="aff"
)

TD_T_cog = correlate_specific_psd(
    TD,
    segment="T",
    domain="cog"
)

TD_T_aff = correlate_specific_psd(
    TD,
    segment="T",
    domain="aff"
)

In [36]:
SAVE_DIR = (
    BASE
    / "EF_ToM_specific_PSD_20"
)

SAVE_DIR.mkdir(
    exist_ok=True
)

results = {
    "ASD_H_cog": ASD_H_cog,
    "ASD_H_aff": ASD_H_aff,
    "ASD_T_cog": ASD_T_cog,
    "ASD_T_aff": ASD_T_aff,

    "TD_H_cog": TD_H_cog,
    "TD_H_aff": TD_H_aff,
    "TD_T_cog": TD_T_cog,
    "TD_T_aff": TD_T_aff,
}

for name, df in results.items():

    df.to_csv(
        SAVE_DIR / f"{name}.csv",
        index=False
    )

In [47]:
from pathlib import Path

# EF_ROOTS = [
#     Path(r"H:\ASD projoect\EF\ASD"),
#     Path(r"H:\ASD projoect\EF\TD"),
# ]

# DELETE_FILES = [
#     "filtered_epoch.npy",
#     "ica_epoch.npy",
# ]

EF_ROOTS = [
    Path(r"H:\ASD projoect\ToM\ASD"),
    Path(r"H:\ASD projoect\ToM\TD"),
]

DELETE_FILES = [
    "filtered_hearing_epoch.npy",
    "filtered_thinking_epoch.npy",
    "ica_hearing_epoch.npy",
    "ica_thinking_epoch.npy"
]

# Delete files in the middle

In [48]:
files_to_delete = []

for root in EF_ROOTS:
    for file_name in DELETE_FILES:
        files_to_delete.extend(
            root.rglob(file_name)
        )

print("Files found:", len(files_to_delete))

for f in files_to_delete:
    print(f)

Files found: 160
H:\ASD projoect\ToM\ASD\3011a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3021a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3031a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3041a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3051a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3061a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3071a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3131a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3081a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3091a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3101a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3111a_CV_processed_numpy\filtered_hearing_epoch.npy
H:\ASD projoect\ToM\ASD\3121a_CV_processed_numpy\filtered_hearing_epoch

In [49]:
deleted = 0
failed = []

for f in files_to_delete:

    try:
        f.unlink()
        deleted += 1

    except Exception as e:
        failed.append((f, str(e)))

print("Deleted:", deleted)
print("Failed:", len(failed))

for f, error in failed:
    print("FAILED:", f)
    print(" ", error)

Deleted: 160
Failed: 0
